In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

## Статистически анализ на производствените цикли (MC15 - 45 liter box)

Данните от датчиците на шприцмашината за записаните 24 цикъла показват висок интензитет на стабилност, но детайлният статистически контрол открива специфични отклонения, които носят риск от дефектни компоненти.


### 1. Статистически профил на производствения процес

Въз основа на извадката, основните параметри на топлообмена и динамиката на впръскване показват следните средни стойности и нива на стабилност:

*   **Време за запълване (*Filling time*):** Стабилно около **0.90 секунди**. Общият разнос е минимален (от 0.89 до 0.91 сек), което показва постоянна скорост на впръскване.
*   **Пиково налягане (*Pressure peak*):** Средната стойност е **121.78 bar**. Тесният диапазон на вариация доказва хомогенност и стабилен вискозитет на стопилката от Полипропилен.
*   **Позиция на възглавницата (*Cushion position*):** Средно ниво от **12.55 mm**. Поддържането на стабилна възглавница в края на дозирането гарантира, че налягането на задържане (*Hold-on pressure*) се предава ефективно, предотвратявайки появата на всмукнатини и обемно свиване.



### 2. Идентифицирани аномалии и технологичен риск

Математическият анализ улавя три конкретни цикъла, които излизат извън границите на нормалното статистическо разпределение и носят вероятност за генериране на дефектна продукция:

####  Цикъл 17928 — Висок риск от геометричен дефект (Препълване)
*   **Регистрирано отклонение:** Отчетено е най-високото пиково налягане (**122.40 bar**), максимален въртящ момент на шнека (**1004 Nm**) и голямо отклонение при началната позиция на пластификация (**14.0 mm** спрямо средно 12.68 mm).
*   **Ефект върху качеството:** Прекомерното налягане и обем материал в кухината могат да доведат до **препълване на матрицата (Flash)** — излизане на пластмасови мустаци по деленията на инструмента или надвишаване на теглото на кутията.

#### Цикъл 17919 — Риск от недопълване или шупли
*   **Регистрирано отклонение:** Началното налягане на дозиране спада рязко до **47.40 bar** (при нормални ~51.70 bar), а остатъчната възглавница изтънява до критичния си минимум за серията — **11.60 mm**.
*   **Ефект върху качеството:** Риск от вкарване на въздух в кухината, което води до повърхностни линии, **вътрешни шупли** или недопълване в най-отдалечените зони на детайла.

####  Цикъл 17937 — Леко кинематично отклонение
*   **Регистрирано отклонение:** Позицията на превключване към налягане на задържане (*Switch-over position*) е паднала до **52.90 mm** (при фиксирани 53.00 mm при всички останали цикли).


### 3. Заключение и оценка на вероятността за брак

*   **Пригодност на процеса:** Производствената линия показва стабилност на ниво от **91.6%** (22 от 24 цикъла са в перфектни статистически граници).
*   **Вероятност за некачествена продукция:** Математическата вероятност за дефектен детайл в тази партида се оценява на **~8.3%**, визирайки конкретно цикли **17928** и **17919**.



In [10]:
df = pd.read_csv("data/mc15.csv")
# Премахване на средните стойности изчиляване на статистическите показатели
df_cycles = df[df['Cycle No.'] != 'Average'].copy()
for col in df_cycles.columns:
    if col != 'Cycle No.':
        df_cycles[col] = pd.to_numeric(df_cycles[col])
summary = df_cycles.describe().T[['mean', 'std', 'min', 'max']]
print(summary.to_string())

# Проверете за аномалии или отклоняващи се стойности
outliers_list = []
for col in df_cycles.columns:
    if col != 'Cycle No.':
        m = df_cycles[col].mean()
        s = df_cycles[col].std()
        if s > 0:
            out_count = ((df_stats := df_cycles[col]) < m - 3*s) | (df_stats > m + 3*s)
            if out_count.sum() > 0:
                anomalies = df_cycles.loc[out_count, ['Cycle No.', col]]
                for _, row in anomalies.iterrows():
                    outliers_list.append({
                        "Параметър": col,
                        "Номер на цикъл": int(float(row['Cycle No.'])),
                        "Измерена стойност": row[col]
                    })
if outliers_list:
    df_outliers = pd.DataFrame(outliers_list)
    print("\nОТКРИТИ АНОМАЛИИ В ПРОЦЕСА:")
    display(df_outliers.style.hide(axis="index"))
else:
    print("\nНяма открити аномалии извън 3-sigma границите.")




                                         mean        std     min      max
Filling time (s)                     0.904167   0.009286    0.89     0.91
Hold-on pressure (bar)              55.083333   0.038069   55.00    55.10
Switch-over position (mm)           52.995833   0.020412   52.90    53.00
Switch-over pressure (bar)         121.841667   0.230154  121.50   122.50
Injection start position (mm)      214.279167   0.241335  213.90   214.80
Pressure peak position (mm)         54.170833   0.227423   53.90    54.60
Pressure peak (bar)                121.779167   0.232153  121.40   122.40
Cushion position (mm)               12.545833   0.395605   11.60    13.50
Cushion minimum position (mm)       12.420833   0.418048   11.40    13.40
Plasticising time (s)               14.950833   0.317708   14.47    15.66
Plasticising position (mm)         200.095833   0.157367  199.80   200.30
Backpressure (bar)                   9.662500   0.049454    9.60     9.70
Rotation speed (rpm)               250

Параметър,Номер на цикъл,Измерена стойност
Switch-over position (mm),17937,52.900000
Plasticising start position (mm),17928,14.000000
Plasticising start pressure (bar),17928,47.400000
Plasticising start pressure (bar),17919,47.400000


### Описание на статистическите показатели (Метрики на процеса)

За оценка на стабилността на машината при 45-литровата кутия се използват следните четири дескриптивни статистически показателя:

1. **Средна стойност (`mean`):** Дефинира технологичния център на процеса. Показва средното аритметично ниво, около което флуктуират параметрите.
2. **Стандартно отклонение (`std`):** Измерва средноквадратичното разсейване на данните. Служи като директен измерител за стабилността на шприцмашината — по-ниската стойност означава по-висока повтаряемост между отделните производствени цикли.
3. **Минимална стойност (`min`):** Най-ниската регистрирана точка на параметъра в извадката, дефинираща долната граница на реалното разпределение.
4. **Максимална стойност (`max`):** Най-високата регистрирана точка на параметъра, отразяваща върховите пикове и натоварвания в системата.
